# US Business Dynamics
### _An Exploratory Data Analysis of Business Growth and Change in the US_

### Project Overview

This project explores patters of US business activity using the Census Bureau's Business Dynamics Statistics dataset. This data provides measures of business activities such as employment, job creation, job desctruction, establishment openings, establishment closings, firm start-ups, and firm shut-downs. 
Its purpose is to perform EDA on a large, real-world dataset and show that process from load to insight. 

### Primary Question

**What patterns emerge when running data analysis on the BDS dataset provided by the US Census Bureau?**

### Analysis Guide

This project will progress through:
1. Data acquisition and inspection
2. Data cleaning and quality assessment
3. Feature engineering
4. Univariate exploratory analysis
5. Multivariate exploratory analysis
6. Data visualization
7. Interpretation of key findings
8. Tableau visualization

## 1. Data Acquisition

**Data Source:** Us Census Bureau - Business Dynamics Statistics (BDS) dataset

The BDS provides annual measures of businesss activity in the US including employment, job creation, job desctruction, establishment openings, establishment closings, firm start-ups, and firm shut-downs. The dataset will focus on the State by Sector section of data to provide observations across three insightful dimensions:
- **Time:** Annual observations from 1978 through 2023
- **Geography:** US States
- **Industry:** NAICS industry sectors

Structurally, this allows business activity to be displayed and examined over time, industry, and geographic area, enriching the data with more meaning and depth. 

**Source Links (URLs)**

US Census Bureau Business Dynamics Statistics:
https://www.census.gov/programs-surveys/bds.html

BDS Datasets:
https://www.census.gov/programs-surveys/bds/data.Datasets.html

BDS Codes and Glossary:
https://www.census.gov/programs-surveys/bds/documentation.html

https://www.census.gov/library/reference/code-lists/ansi/ansi-codes-for-states.html

https://www.census.gov/programs-surveys/economic-census/year/2022/guidance/understanding-naics.html

_Import necessary Python packages_

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Dataset URL

https://www2.census.gov/programs-surveys/bds/tables/time-series/2023/bds2023_st_sec.csv

_The raw data in csv form will be loaded into a pandas dataframe_

In [3]:
df = pd.read_csv("data/bds2023_st_sec.csv")

_Before running any analysis, we will check the dataset's dimensions and size..._

In [4]:
df.shape

(44574, 27)

_...and inspect its headers to identify the types of business metrics available for analysis. Note, while df.head() is useful to explore a dataset's dimensions, we do not use it here because pandas cannot display all headers for this specific dataset._

In [5]:
df.columns.tolist()

['year',
 'st',
 'sector',
 'firms',
 'estabs',
 'emp',
 'denom',
 'estabs_entry',
 'estabs_entry_rate',
 'estabs_exit',
 'estabs_exit_rate',
 'job_creation',
 'job_creation_births',
 'job_creation_continuers',
 'job_creation_rate_births',
 'job_creation_rate',
 'job_destruction',
 'job_destruction_deaths',
 'job_destruction_continuers',
 'job_destruction_rate_deaths',
 'job_destruction_rate',
 'net_job_creation',
 'net_job_creation_rate',
 'reallocation_rate',
 'firmdeath_firms',
 'firmdeath_estabs',
 'firmdeath_emp']

_We will inspect the dataset's structure, data types and non-null counts to determine how each variable is interpreted during import and to identify fields that may require more processing._

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44574 entries, 0 to 44573
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   year                         44574 non-null  int64
 1   st                           44574 non-null  int64
 2   sector                       44574 non-null  str  
 3   firms                        44574 non-null  str  
 4   estabs                       44574 non-null  str  
 5   emp                          44574 non-null  str  
 6   denom                        44574 non-null  str  
 7   estabs_entry                 44574 non-null  str  
 8   estabs_entry_rate            44574 non-null  str  
 9   estabs_exit                  44574 non-null  str  
 10  estabs_exit_rate             44574 non-null  str  
 11  job_creation                 44574 non-null  str  
 12  job_creation_births          44574 non-null  str  
 13  job_creation_continuers      44574 non-null  str  
 14  j

_Observation:_

_The first three columns in our dataframe represent yea, state, and sector. These do not need to be converted to numeric form as we will convert them to their respective indicators later as a cleaning step. Furthermore, most of the raw BDS data represents numeric counts or rates (int64), but pandas imported 25 of the 27 columns as strings (str), which may hamper our data analysis. Before we convert these strings to numbers, we will investigate further to see why these values are seen as strings._

_Assumption:_

_We can assume the column header "firms" should represent subsequent data as the numeric value of firms._ 

_Action:_

_Since we have an exorbitant amount of data to sift through, we will first see a light representation of how often a str type value appears rather than int64._

In [7]:
df["firms"].value_counts(dropna=False).head(20)

firms
39     49
73     47
74     47
163    44
44     44
78     42
103    42
D      42
125    41
60     41
90     41
105    41
131    41
69     40
89     40
104    40
48     40
85     40
93     39
161    39
Name: count, dtype: int64

_Then, we will create a temporary dataframe from the original while converting all data types to int64 (.to_numeric), force all non int64 values into NaN values (errors=coerce), as well as sift through the numeric data for anomalies to show us why this column may be imported as str type and how often those anomalies occur._

In [8]:
firms_numeric = pd.to_numeric(df["firms"], errors="coerce")

df.loc[firms_numeric.isna(), "firms"].value_counts(dropna=False)

firms
D    42
Name: count, dtype: int64

_The output of the cell above lists "D" as the culprit as to why our data is of str type and not numeric. In perusing the BDS Glossary, we can see:_
> Disclosure Suppression – Disclosure suppressions are made when a cell has too few firms. Cells suppressed due to containing too few firms will appear as “D”.

_It is always a good idea to take into consideration auxiliary supporting documentation when assessing datasets for meaning._

_All values in the "firms" column designated as "D" are therefore **NOT** zeros, rather they are suppressed and cannot be inferred through data analysis. We will check to see if any other column within our dataframe that is of data type str contains indicators we can look up in our glossary._

_Anomaly scrounging:_

_We will start with an empty dictionary, we will convert all columns in our dataframe to numeric starting with the fourth column (df.columns[3:]) as we know the first three don't necessarily need numeric conversion, identify which ones cannot convert, load those values into the empty dictionary and then we will display its contents._

In [9]:
BDS_indicators = {}

for column in df.columns[3:]:
    numeric_version = pd.to_numeric(df[column], errors="coerce")

    invalid_values = (
        df.loc[numeric_version.isna(), column]
        .value_counts()
        .to_dict()
    )

    if invalid_values:
        BDS_indicators[column] = invalid_values

BDS_indicators

{'firms': {'D': 42},
 'estabs': {'D': 42},
 'emp': {'D': 42},
 'denom': {'D': 43},
 'estabs_entry': {'D': 445},
 'estabs_entry_rate': {'D': 445, 'N': 1},
 'estabs_exit': {'D': 542},
 'estabs_exit_rate': {'D': 542, 'N': 1},
 'job_creation': {'D': 42},
 'job_creation_births': {'D': 445},
 'job_creation_continuers': {'D': 51},
 'job_creation_rate_births': {'D': 445, 'N': 1},
 'job_creation_rate': {'D': 42, 'N': 1},
 'job_destruction': {'D': 43},
 'job_destruction_deaths': {'D': 542},
 'job_destruction_continuers': {'D': 51},
 'job_destruction_rate_deaths': {'D': 542, 'N': 1},
 'job_destruction_rate': {'D': 43, 'N': 1},
 'net_job_creation': {'D': 43},
 'net_job_creation_rate': {'D': 43, 'N': 1},
 'reallocation_rate': {'D': 42, 'N': 1},
 'firmdeath_firms': {'D': 1339},
 'firmdeath_estabs': {'D': 1339},
 'firmdeath_emp': {'D': 1339}}

_In looking through the data ouput, we see that str values "D" and "N" appear frequently. From the glossary:_
> Rate Not Available – Rates that cannot be calculated due to a denominator of ‘0’ will appear as “N”.

_These indicators are also not evenly distributed through the dataset, with some columns containing more or less than others of these indicators. We, again, will **NOT** consider these values as zeros._

_Our dataset is ready to clean._

## 2. Data Cleaning

In the previous exercise, we found that many values were listed as str type because they represented unavailable data and utilized an alphabetic indicator code instead of a numeric value. To progress with our exploratory data analysis, we should find a way to convert these values into data types that can be analyzed.

_We will go forward with a copy of the dataframe so as to not irreversably alter the original. We will further sequester the columns to convert into numeric form by creating a new variable containing those columns and disregarding the first three (df_clean.columns[3:])._

In [31]:
df_clean = df.copy()

In [32]:
columns_tonumeric = df_clean.columns[3:]

_Now, we will use the variable and undergo conversion from whatever datatype they currently exist as to numeric. All errors (alphabetic values/strings) will be coerced into computable types ("NaN")._

In [33]:
for column in columns_tonumeric:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

_Let's check that the conversion worked by listing the datatypes contained within our copied dataframe._

In [34]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 44574 entries, 0 to 44573
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   year                         44574 non-null  int64  
 1   st                           44574 non-null  int64  
 2   sector                       44574 non-null  str    
 3   firms                        44532 non-null  float64
 4   estabs                       44532 non-null  float64
 5   emp                          44532 non-null  float64
 6   denom                        44531 non-null  float64
 7   estabs_entry                 44129 non-null  float64
 8   estabs_entry_rate            44128 non-null  float64
 9   estabs_exit                  44032 non-null  float64
 10  estabs_exit_rate             44031 non-null  float64
 11  job_creation                 44532 non-null  float64
 12  job_creation_births          44129 non-null  float64
 13  job_creation_continuers    

From the logic check above, we can see that the columns we identified as needing conversion to numeric data type have all become floats which are computable. We still need to deal with the columns "st" and "sector". 

In the explanatory links to the BDS coding above, we can find mappings for the numeric FIPS and NAICS codes contained in our columns representing state and sector. 

_We will map the state codes to state names._

In [35]:
state_codes = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    56: "Wyoming"
}

In [36]:
df_clean["st"] = df_clean["st"].map(state_codes)

_Next, we will do the same for mapping the sector codes to individually identifiable sectors._

In [37]:
sector_codes = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44-45": "Retail Trade",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management and Remediation Services",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services (except Public Administration)"
}

In [38]:
df_clean["sector"] = df_clean["sector"].map(sector_codes)

_Let's now validate that the mappings have occured and that we have not inadvertently created any "Nan" values by letting values that were not included in the code listings through the cracks._

In [45]:
df_clean[["year", "st","sector"]].head(2)

,year,st,sector
0,1978,Alabama,"Agriculture, Forestry, Fishing and Hunting"
1,1978,Alabama,"Mining, Quarrying, and Oil and Gas Extraction"


In [44]:
print(df_clean["st"].isna().sum())

print(df_clean["sector"].isna().sum())

0
0


We can see that the mapping has occurred successfully, and that there are zero "Nan" values for both the "st" and "sector" column values. We can check the "year" column to figure out if there are any missing years. This will prevent any confounding observations when undergoing analysis in the near future.

_We will check for year continuity by subtracting the difference between the maximum and minimum listed year in the column's values and adding one value to insure we are counting all years in that range (+ 1) from the amount of unique years (["year"].unique())._

In [48]:
expected_years = set(range(
    df_clean["year"].min(),
    df_clean["year"].max() + 1
))

actual_years = set(df_clean["year"].unique())

expected_years - actual_years

set()

This yields an expty set, which we can infer means there are no missing values betwee the minimum and maximum listed years in our dataframe. 

Now we can search for missing rows and make sure all rows are unique. This gives us assurance that there are no duplicate rows. 

_To search for missing rows, we count the unique amount of years, states, and sectors in our data as a total dimension, and then compare that with our current row count._

In [55]:
n_years = df_clean["year"].nunique()
n_states = df_clean["st"].nunique()
n_sectors = df_clean["sector"].nunique()

print("Years = " + str(n_years))
print("States = " + str(n_states))
print("Sectors = " + str(n_sectors))
print("Dimension = " + str(n_years * n_states * n_sectors))

Years = 46
States = 51
Sectors = 19
Dimension = 44574


In [56]:
row_dimension = n_years * n_states * n_sectors
row_actual = len(df_clean)

print("Expected rows = " + str(row_dimension))
print("Actual rows = " + str(row_actual))

Expected rows = 44574
Actual rows = 44574


While this does affirm our idea of row counts matching, it does not tell us if any of the rows are duplicates. To find out if our data contains duplicates, we can use pandas' native method (.duplicated()).

_We will use the method to find duplicates within the dimension defined above ("year" * "st" * "sector"). A sum of these duplicates should equal zero._

In [57]:
duplicate_count = df_clean.duplicated(
    subset=["year", "st", "sector"]
).sum()

duplicate_count

np.int64(0)

A result of zero shows that our dataset has no duplicates.

**Duplicate rows are a common occurence in datasets that will undergo analysis. We have escaped tragedy here by finding none, but had we found any, we could have used pandas' native duplicate dropping method (.drop_duplicates()) to remove them and continue forward.**

_Let's display a visual representation of all the value counts and dimensions we have validated thus far for reference._

In [61]:
print(f"Years: {n_years}")
print(f"States: {n_states}")
print(f"Sectors: {n_sectors}")
print(f"Expected rows: {row_dimension:,}")
print(f"Actual rows: {row_actual:,}")
print(f"Duplicate Year x State x Sector combinations: {duplicate_count}")

Years: 46
States: 51
Sectors: 19
Expected rows: 44,574
Actual rows: 44,574
Duplicate Year x State x Sector combinations: 0
